In [12]:
import uproot
import os
import glob
import matplotlib.pyplot as plt
import mplhep as hep
import seaborn as sns
import ROOT
import pandas as pd
from scipy import stats
import numpy as np
from scipy.optimize import minimize, curve_fit
from scipy.stats import moment
import json
from itertools import combinations  

In [2]:
# Importing translation dictionary
translationFilePath = "/work/niharrin/t35/CMSSW_14_1_0_pre4/src/flashggFinalFit/Bootstrap/translation.json"
with open(translationFilePath) as translationFile:
    translation = json.load(translationFile)

In [3]:
def pois_untrimmed(toyDir_, poi_list_):
    pois = {}

    for current_poi in poi_list_:

        pois[current_poi] = []

    for i in range(len(glob.glob(os.path.join(toyDir_, "toy_*")))):

        if i%100==0:
            print(f"Processing fit_{i}")

        try:
            current_root_files = uproot.open(f"{toyDir_}/toy_{i}/higgsCombinefirstStep.MultiDimFit.mH125.38.root")
        except:
            print(f"Skipping fit_{i}: Required scan files not found")
            continue

        for j, current_poi in enumerate(poi_list_):

            try: 
                current_tree = current_root_files["limit"]
            except:
                print(f"Skipping fit_{i}: Tree 'limit' not found in ROOT file")
                continue

            current_limit_values = current_tree[current_poi].array()

            try:
                if current_limit_values[0]<-4:
                    print("Value out of boundary", current_limit_values[0], i)
            except:
                if j == 0:
                    print("Empty ROOT file for bootstrap: ", i)
                continue

            pois[current_poi].append(float(current_limit_values[0]))
    return pois

In [4]:
def pois_trimmed(toyDir_, poi_list_, trimming_value_left_, trimming_value_right_):
    pois = {}

    for current_poi in poi_list_:

        pois[current_poi] = []

    for i in range(len(glob.glob(os.path.join(toyDir_, "toy_*")))):

        if i%100==0:
            print(f"Processing fit_{i}")

        try:
            current_root_files = uproot.open(f"{toyDir_}/toy_{i}/higgsCombinefirstStep.MultiDimFit.mH125.38.root")
        except:
            print(f"Skipping fit_{i}: Required scan files not found")
            continue

        kill_event = False

        # First loop to check if there are outliers (outliers are events smaller than -4 and bigger than 4)
        for j, current_poi in enumerate(poi_list_):
            try: 
                current_tree = current_root_files["limit"]
            except:
                print(f"Skipping fit_{i}: Tree 'limit' not found in ROOT file")
                continue

            current_limit_values = current_tree[current_poi].array()

            try:
                if (current_limit_values[0] > trimming_value_right_) or (current_limit_values[0] < trimming_value_left_):
                    kill_event = True
                    break
            except:
                if j == 0:
                    print("Empty ROOT file for bootstrap: ", i)
                continue

        if kill_event:
            print(f"Skipping fit_{i} with {current_limit_values}: Outlier found")
            continue

        for j, current_poi in enumerate(poi_list_):

            try: 
                current_tree = current_root_files["limit"]
            except:
                print(f"Skipping fit_{i}: Tree 'limit' not found in ROOT file")
                continue

            current_limit_values = current_tree[current_poi].array()

            try:
                if current_limit_values[0]<-4:
                    print("Value out of boundary", current_limit_values[0], i)
            except:
                if j == 0:
                    print("Empty ROOT file for bootstrap: ", i)
                continue

            pois[current_poi].append(float(current_limit_values[0]))
    return pois

In [5]:
def create_json_untrimmed(variable_, toyDir_, poi_list_, extra_name=""):
    # Loop through all fit directories (fit_0, fit_1, etc.)
    if not os.path.exists(f'pois_untrimmed_{variable_}{extra_name}.json'):
        pois = pois_untrimmed(toyDir_, poi_list_)
        # Save the POIs to a JSON file
        with open(f'pois_untrimmed_{variable_}{extra_name}.json', 'w') as f:
            json.dump(pois, f)
    else:
        # Load the POIs from the JSON file
        with open(f'pois_untrimmed_{variable_}{extra_name}.json', 'r') as f:
            pois = json.load(f)

    return pois


In [6]:

def gaus(x, amp, mu, sigma):
    return amp * np.exp(-(x - mu)**2 / (2 * sigma**2))

def create_json_trimmed(variable_, toyDir_, pois_untrimmed_, poi_list_, subfolder_, extra_name=""):

    # Now we have to fit a gaussian core to all the categories to get the values where we cut. z is the number of standard deviations we want to cut away
    trimming_value_left = 0
    trimming_value_right = 0
    largest_sigma = 0
    mu_to_largest_sigma = 0
    z = 4
    for i, current_poi in enumerate(poi_list_):
        r = np.array(pois_untrimmed_[current_poi])
        mean = np.mean(r)
        s = np.std(r)

        counts, bin_edges = np.histogram(r, bins=50, density=True)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        popt, _ = curve_fit(gaus, bin_centers, counts, p0=[1, mean, s])
        amp_fit, mu_fit, sigma_fit = popt

        if i == 0:
            largest_sigma = sigma_fit
            mu_to_largest_sigma = mu_fit
        else:
            if (sigma_fit > largest_sigma):
                largest_sigma = sigma_fit
                mu_to_largest_sigma = mu_fit

        if (not os.path.exists(f"./Plots/{variable_}/{subfolder_}{extra_name}")) & (f"./Plots/{variable_}/{subfolder_}{extra_name}"!=""):
            os.makedirs(f"./Plots/{variable_}/{subfolder_}{extra_name}")

        plt.figure()
        plt.hist(r, bins=30, density=True, alpha=0.6, label='Histogram')
        plt.plot(bin_centers, gaus(bin_centers, *popt), color='red', label='Fitted Gaussian')
        plt.legend()
        plt.xlabel('Value')
        plt.ylabel('Density')
        plt.title('Gaussian Fit to Data Histogram')
        plt.savefig(f"./Plots/{variable_}/{subfolder_}{extra_name}/gaussian_fit_{current_poi}.png")
        plt.close()

    trimming_value_left = mu_to_largest_sigma - z*largest_sigma
    trimming_value_right = mu_to_largest_sigma + z*largest_sigma

    print("Trimming values: ", trimming_value_left, trimming_value_right)

    if trimming_value_left > trimming_value_right:
        trimming_value_left, trimming_value_right = trimming_value_right, trimming_value_left

    if not os.path.exists(f'pois_trimmed_{variable_}{extra_name}.json'):
        pois = pois_trimmed(toyDir_, poi_list_, trimming_value_left, trimming_value_right)
        # Save the POIs to a JSON file
        with open(f'pois_trimmed_{variable_}{extra_name}.json', 'w') as f:
            json.dump(pois, f)
    else:
        # Load the POIs from the JSON file
        with open(f'pois_trimmed_{variable_}{extra_name}.json', 'r') as f:
            pois = json.load(f)
    
    return pois



In [40]:
def plot_all_individual_correlation(pois_, pois_other_, poi_list_, variable_):

    unique_pairings = list(combinations(poi_list_, 2))

    for current_tuple in unique_pairings:
        r_1, r_2 = current_tuple

        plot_individual_correlation(pois_[r_1], pois_other_[r_1], pois_[r_2], pois_other_[r_2], r_1, r_2, np.corrcoef(pois_[r_1], pois_[r_2])[0,1], np.corrcoef(pois_other_[r_1], pois_other_[r_2])[0,1], folder=f"Plots/{variable_}")

def plot_individual_correlation(x_vals, x_vals2, y_vals, y_vals2, x_name, y_name, rho_, rho2_, folder=""):
    # rho is here a number
    if (not os.path.exists(folder)) & (folder!=""):
        os.makedirs(folder)
    plt.style.use(hep.style.CMS)
    _, ax = plt.subplots(figsize=(12, 7))
    hep.cms.label('Preliminary', data=False, lumi=9.5, com=13.6)
    ax.scatter(x_vals, y_vals, color="black", marker="x", label="Manual Toys")
    ax.scatter(x_vals2, y_vals2, color="red", label="Combine Toys")
    ax.set_xlabel(translation[x_name])
    ax.set_ylabel(translation[y_name])
    bbox_props = dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="black")
    ax.text(0.05, 0.175, f"$\\rho(Manual Toys) = {rho_:.3f}$", transform=ax.transAxes, bbox=bbox_props, fontsize=18)
    ax.text(0.05, 0.075, f"$\\rho(Combine Toys) = {rho2_:.3f}$", transform=ax.transAxes, bbox=bbox_props, fontsize=18)
    leg = ax.legend(fancybox=True, loc="lower right", frameon=True)
    leg.get_frame().set_alpha(0.9)  # Now this actually applies
    plt.savefig(os.path.join(folder, f"{x_name}_vs_{y_name}.pdf"), bbox_inches='tight')
    plt.savefig(os.path.join(folder, f"{x_name}_vs_{y_name}.png"), bbox_inches='tight')
    plt.close()


In [8]:
plot_dir = "Plots"
os.makedirs(plot_dir, exist_ok=True)

In [9]:
# Create JSON directory
json_dir = "json"
os.makedirs(json_dir, exist_ok=True)

# Load "our" toys
variable = "PTH"
sample_dir = '/pnfs/psi.ch/cms/trivcat/store/user/niharrin/ntuples/midRun3/samples/2025_07_17_powheg/finalfits'
main_dir = os.path.join(sample_dir, variable, "Combine", f"runFits_{variable}")
toyDir = os.path.join(main_dir, "toyFit")

poi_list = ["r_PTH_0p0_15p0", "r_PTH_15p0_30p0", "r_PTH_30p0_45p0", "r_PTH_45p0_80p0", "r_PTH_80p0_120p0", "r_PTH_120p0_200p0", "r_PTH_200p0_350p0", "r_PTH_350p0_10000p0"]

pois_untrimmed_ourToys = create_json_untrimmed(variable, toyDir, poi_list)
pois_ourToys = create_json_trimmed(variable, toyDir, pois_untrimmed_ourToys, poi_list, subfolder_=f"SL_{variable}")

Trimming values:  -5.408250892941592 7.758428066877565


In [10]:
# Load combine toys
combineToy_dir = "/work/niharrin/t35/CMSSW_14_1_0_pre4/src/flashggFinalFit/Replicas/CombineToy_Validation/toys"

pois_untrimmed_combineToys = create_json_untrimmed(variable, combineToy_dir, poi_list, extra_name="_combineToys")
pois_combineToys = create_json_trimmed(variable, combineToy_dir, pois_untrimmed_combineToys, poi_list, subfolder_=f"SL_{variable}", extra_name="_combineToys")

Trimming values:  -5.048050842022782 7.324431154999333


In [41]:
plot_all_individual_correlation(pois_ourToys, pois_combineToys, poi_list, variable)

In [67]:
for current_poi in poi_list:
    r_ourToy = pois_ourToys[current_poi]
    r_combineToy = pois_combineToys[current_poi]
    
    # 1. Find combined min and max
    combined_min = min(min(r_ourToy), min(r_combineToy))
    combined_max = max(max(r_ourToy), max(r_combineToy))

    # 2. Make shared bin edges (30 bins)
    bin_edges = np.linspace(combined_min, combined_max, 31)
    
    plt.style.use(hep.style.CMS)
    plt.figure()
    hep.cms.label('Preliminary', data=False, lumi=9.5, com=13.6)
    plt.hist(r_ourToy, bins=bin_edges, density=True, alpha=0.6, label='Manual Toys', color='black')
    plt.hist(r_combineToy, bins=bin_edges, density=True, alpha=0.6, label='Combine Toys', color='red')
    plt.xlabel(translation[current_poi])
    plt.legend()
    plt.savefig(os.path.join("Plots", variable, f"{current_poi}_histo.pdf"), bbox_inches='tight')
    plt.savefig(os.path.join("Plots", variable, f"{current_poi}_histo.png"), bbox_inches='tight')
    plt.close()
    